# 07. Baseline variants check

Compare three response baseline variants on visits against the S learner on the 1M development split.

In [1]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from uplift.data import TREATMENT, make_splits, take
from uplift.learners import BASE_PARAMS, _prob
from uplift.metrics import bootstrap_metrics, qini_coefficient, paired_difference

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
RESULTS = ROOT / "results"

df = pd.read_parquet(ROOT / "data" / "criteo.parquet")
test_idx, pool_idx = make_splits(len(df))
dev = take(df, pool_idx[:1_000_000])
held = take(df, test_idx)

kept = json.load(open(RESULTS / "kept_features.json"))
params = json.load(open(RESULTS / "best_params.json"))["visit"]
print("Dev shape:", dev.shape, "Held shape:", held.shape)
print("Kept features:", kept)
print("Visit params:", params)

Dev shape: (1000000, 16) Held shape: (2000000, 16)
Kept features: ['f0', 'f2', 'f3', 'f4', 'f6', 'f7', 'f8', 'f9', 'f10', 'f11']
Visit params: {'num_leaves': 15, 'learning_rate': 0.05, 'n_estimators': 200, 'min_child_samples': 200}


In [2]:
X_dev, y_dev, t_dev = dev[kept], dev["visit"], dev[TREATMENT]
X_held, y_held, t_held = held[kept], held["visit"].to_numpy(), held[TREATMENT].to_numpy()

model_a = LGBMClassifier(**{**BASE_PARAMS, **params}).fit(X_dev, y_dev)
model_b = LGBMClassifier(**{**BASE_PARAMS, **params}).fit(X_dev[t_dev == 1], y_dev[t_dev == 1])
model_c = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=42)).fit(X_dev, y_dev)

score_a = _prob(model_a, X_held)
score_b = _prob(model_b, X_held)
score_c = _prob(model_c, X_held)

scores_1m = pd.read_parquet(RESULTS / "scores_1m.parquet")
aligned = scores_1m.loc[held.index]
score_s = aligned["s_visit"].to_numpy()

max_diff = float(np.max(np.abs(score_a - aligned["response_visit"].to_numpy())))
print(f"Max diff vs response_visit in scores_1m.parquet: {max_diff:.2e}")
assert max_diff < 1e-4, f"Variant A mismatch: {max_diff}"

qini_5m = pd.read_csv(RESULTS / "qini_results_visit_5m.csv")
print("5M response Qini (reference):", qini_5m.loc[qini_5m.iloc[:, 0] == "response", "qini_coef"].values[0])
print("Variant A 1M Qini:", round(qini_coefficient(y_held, t_held, score_a), 6))

Max diff vs response_visit in scores_1m.parquet: 2.98e-08
5M response Qini (reference): 0.003402
Variant A 1M Qini: 0.003407


In [3]:
scores_dict = {
    "A_all_rows_lgbm": score_a,
    "B_treated_only_lgbm": score_b,
    "C_logistic_regression": score_c,
    "S_learner": score_s,
}

boot = bootstrap_metrics(y_held, t_held, scores_dict, metric=qini_coefficient, n_boot=200, seed=42)

rows = []
for name in ["A_all_rows_lgbm", "B_treated_only_lgbm", "C_logistic_regression"]:
    pt = qini_coefficient(y_held, t_held, scores_dict[name])
    ci_lo = float(boot[name].quantile(0.025))
    ci_hi = float(boot[name].quantile(0.975))
    s_pt = qini_coefficient(y_held, t_held, score_s)
    pd_res = paired_difference(boot, "S_learner", name, s_pt, pt)
    vals = list(pd_res.values()) if isinstance(pd_res, dict) else list(pd_res)
    rows.append({
        "variant": name,
        "qini_coef": pt,
        "ci_lo": ci_lo,
        "ci_hi": ci_hi,
        "s_minus_variant": float(vals[0]),
        "diff_lo": float(vals[1]),
        "diff_hi": float(vals[2]),
        "p_one_sided": float(vals[3]),
    })

out = pd.DataFrame(rows)
out.to_csv(RESULTS / "baseline_variants_visit.csv", index=False)
out.round(6)

,variant,qini_coef,ci_lo,ci_hi,s_minus_variant,diff_lo,diff_hi,p_one_sided
0,A_all_rows_lgbm,0.003407,0.003154,0.003737,0.000168,0.000043,0.000277,0.009950
1,B_treated_only_lgbm,0.003400,0.003150,0.003731,0.000174,0.000045,0.000283,0.009950
2,C_logistic_regression,0.003242,0.002966,0.003590,0.000333,0.000178,0.000469,0.004975


## Conclusion

The S learner still beats every baseline variant on visits, with positive paired differences across all three comparisons. None of these variants is stronger than our baseline.